In [0]:
# A tabela precisa estar com 20 linhas
print("Tabela Atual")
display(spark.table("fin_prod.gold.dim_categoria"))

# Histórico de versões
print("Histórico da Tabela")
display(spark.sql("DESCRIBE HISTORY fin_prod.gold.dim_categoria"))

In [0]:
import pyspark.sql.functions as F

# Simulação do bug: sobrescreve a tabela com só 5 linhas e nome errado
df_corrompido = (spark.table("fin_prod.gold.dim_categoria")
                 .limit(5)
                 .withColumn("nome_categoria", F.lit("ERRO_CORROMPIDO")))

df_corrompido.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fin_prod.gold.dim_categoria")

print("Tabela corrompida de propósito!")
display(spark.table("fin_prod.gold.dim_categoria"))

In [0]:
display(spark.sql("DESCRIBE HISTORY fin_prod.gold.dim_categoria"))

In [0]:
df_boa = (spark.read.format("delta")
          .option("versionAsOf", 1)
          .table("fin_prod.gold.dim_categoria"))

print("=== Como a tabela estava na versão 1 (a boa) ===")
display(df_boa)

In [0]:
# Troque o 2 pela sua versão boa
spark.sql("RESTORE TABLE fin_prod.gold.dim_categoria TO VERSION AS OF 1")

print("Tabela restaurada!")
display(spark.table("fin_prod.gold.dim_categoria"))

In [0]:
display(spark.sql("DESCRIBE HISTORY fin_prod.gold.dim_categoria"))